<a href="https://colab.research.google.com/github/istanranjith175b-commits/Exploring-datasets/blob/main/imputing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [49]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# Your starting dataset
df = pd.DataFrame({
    "Age": [25, np.nan, 30, 45, 22],
    "Salary": [50000, 60000, np.nan, 80000, 45000],
    "Category": ["A", "B", "A", np.nan, "B"],
    "Target": [1, 0, 1, 0, 0] # Example target variable
})

# Separate features (X) and target (y)
X = df.drop(columns=["Target"])
y = df["Target"]

# Define preprocessing for numerical columns (Impute then Scale)
numeric_features = ["Age", "Salary"]
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Define preprocessing for categorical columns (Impute then Encode)
categorical_features = ["Category"]
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine them into a single preprocessor
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# Create the final model pipeline
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

# Fit the entire pipeline safely
model_pipeline.fit(X, y)
print("Pipeline trained successfully without data leakage!")


Pipeline trained successfully without data leakage!


In [52]:
import numpy as np
import pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
# Create sample DataFrame with missing values (NaN)
df = pd.DataFrame(
    {
        "Age": [25, np.nan, 30, 45, 22],
        "Salary": [50000, 60000, np.nan, 80000, 45000],
        "Category": ["A", "B", "A", np.nan, "B"],
    }
)
# 1. Mean Imputation (Numerical)
mean_imputer = SimpleImputer(strategy="mean")
df["Age_Mean"] = mean_imputer.fit_transform(df[["Age"]])
# 2. Median Imputation (Numerical - robust to outliers)
median_imputer = SimpleImputer(strategy="median")
df["Salary_Median"] = median_imputer.fit_transform(df[["Salary"]])
# 3. Mode Imputation (Categorical - most frequent)
mode_imputer = SimpleImputer(strategy="most_frequent")
df["Category_Mode"] = mode_imputer.fit_transform(df[["Category"]]).ravel()
# 4. MICE Imputation (Multivariate Imputation by Chained Equations)
# Select numerical columns for MICE
num_cols = ["Age", "Salary"]
mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice = df[num_cols].copy()
df_mice.iloc[:, :] = mice_imputer.fit_transform(df_mice)
df["Age_MICE"] = df_mice["Age"]
df["Salary_MICE"] = df_mice["Salary"]
print(df)

    Age   Salary Category  Age_Mean  Salary_Median Category_Mode   Age_MICE  \
0  25.0  50000.0        A      25.0        50000.0             A  25.000000   
1   NaN  60000.0        B      30.5        60000.0             B  31.766487   
2  30.0      NaN        A      30.0        55000.0             A  30.000000   
3  45.0  80000.0      NaN      45.0        80000.0             A  45.000000   
4  22.0  45000.0        B      22.0        45000.0             B  22.000000   

    Salary_MICE  
0  50000.000000  
1  60000.000000  
2  57324.583367  
3  80000.000000  
4  45000.000000  
